# Pass 3 CCAST — SMILES → IUPAC reverse validation

**Upload these 3 files to the same CCAST folder:**
- `dataset_ready_for_ccast.xlsx`
- `pass3_ccast_validation.py`
- this notebook

**Uses `chemical-converters` (Hugging Face). Do NOT install or use STOUT.**

## Before Run All
1. **Kernel → Restart Kernel** (required if any earlier run showed CUDA errors)
2. Delete any **old inline cells** from a previous notebook version (STOUT / Java / long loop)
3. **Run → Run All** from the top

## Success check
After the run, the sanity-check cell must show **`Non-empty IUPAC: ~4681/4681`**.
If you see **~49/4681**, CUDA failed again — use `USE_CPU = True` in cell 2.

Many `LOW_SIMILARITY` flags are **normal** (1947 names ≠ modern IUPAC). Empty IUPAC is **not** normal.

In [1]:
import os

# ============ EDIT THESE IF NEEDED ============
USE_CPU = False    # True = slow but reliable (use if GPU keeps failing)
GPU_BATCH = True   # True = 40 rows per fresh subprocess (fixes crash at row ~49)
RESUME = False     # True = continue from ccast_checkpoint.csv after interruption
# ==============================================

INPUT = "dataset_ready_for_ccast.xlsx"
OUTPUT = "dataset_from_ccast.xlsx"
CHECKPOINT = "ccast_checkpoint.csv"

os.environ["TOKENIZERS_PARALLELISM"] = "false"

if USE_CPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    print("Mode: CPU (reliable, slower)")
elif GPU_BATCH:
    print("Mode: GPU batched (40 rows per fresh process)")
else:
    print("Mode: GPU single process (restart kernel before Run All)")

Mode: GPU batched (40 rows per fresh process)


In [2]:
# Install packages — NO STOUT, NO rdkit
!pip install chemical-converters jellyfish openpyxl pandas -q

In [3]:
import importlib.util
import re
from pathlib import Path

# Patch chemicalconverters for Python 3.9 (CCAST uses 3.9)
spec = importlib.util.find_spec("chemicalconverters")
if spec and spec.origin:
    patch_file = Path(spec.origin).parent / "names_converters" / "names_converters.py"
    if patch_file.exists():
        text = patch_file.read_text(encoding="utf-8")
        fixed = re.sub(
            r"->\s*list\[str\]\s*\|\s*tuple\[str,\s*float\]\s*\|\s*str", "", text
        )
        fixed = re.sub(r"->\s*list\[str\]\s*\|\s*str", "", fixed)
        if fixed != text:
            patch_file.write_text(fixed, encoding="utf-8")
            print(f"Patched for Python 3.9: {patch_file}")
        else:
            print("Python 3.9 patch: already applied")
    else:
        print(f"Warning: patch file not found: {patch_file}")
else:
    print("Run the pip cell first, then re-run this cell")

Python 3.9 patch: already applied


In [4]:
from pathlib import Path

if not Path(INPUT).exists():
    raise FileNotFoundError(f"Upload {INPUT} to this folder first")
if not Path("pass3_ccast_validation.py").exists():
    raise FileNotFoundError("Upload pass3_ccast_validation.py to this folder first")

print(f"Input OK: {INPUT}")
print(f"Script OK: pass3_ccast_validation.py")

Input OK: dataset_ready_for_ccast.xlsx
Script OK: pass3_ccast_validation.py


In [5]:
# CUDA health check — skip when USE_CPU=True
if not USE_CPU:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        try:
            torch.cuda.synchronize()
            _t = torch.tensor([1.0], device="cuda:0")
            del _t
            torch.cuda.empty_cache()
            print("CUDA health check: OK")
        except Exception as exc:
            raise RuntimeError(
                f"CUDA is poisoned ({exc}).\n"
                "Kernel -> Restart Kernel, then Run All.\n"
                "Or set USE_CPU = True in cell 2."
            ) from exc
else:
    print("Skipping CUDA check (CPU mode)")

CUDA available: True
GPU: NVIDIA A10
CUDA health check: OK


In [ ]:
# Main run — delegates to pass3_ccast_validation.py
from pathlib import Path
from pass3_ccast_validation import run, run_gpu_batched

if GPU_BATCH and not USE_CPU:
    df = run_gpu_batched(
        Path(INPUT),
        Path(OUTPUT),
        checkpoint_path=Path(CHECKPOINT),
        resume=RESUME,
    )
else:
    df = run(
        Path(INPUT),
        Path(OUTPUT),
        resume=RESUME,
        checkpoint_path=Path(CHECKPOINT),
        force_cpu=USE_CPU,
    )


=== GPU batch rows 0-39 / 4680 ===


2026-06-15 16:57:55.779024: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-15 16:57:56.028347: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-06-15 16:57:56.028380: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-06-15 16:57:56.069949: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-15 16:57:57.073434: W tensorflow/stream_executor/platform/de

Device: cuda
Loading model: knowledgator/SMILES2IUPAC-canonical-small on cuda
Processing rows 0-39 of 4681
  batch checkpoint saved (40 rows total)

=== GPU batch rows 40-79 / 4680 ===


2026-06-15 16:58:15.347537: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-15 16:58:15.601568: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-06-15 16:58:15.601605: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-06-15 16:58:15.644099: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-15 16:58:16.655898: W tensorflow/stream_executor/platform/de

Device: cuda
Loading model: knowledgator/SMILES2IUPAC-canonical-small on cuda
Processing rows 40-79 of 4681
  Checkpoint loaded: 40 rows already done
  CUDA error at row 49 — stopping batch (will retry on CPU)
  batch checkpoint saved (50 rows total)
  GPU batch incomplete (exit 0) — retrying rows 40-79 on CPU


...

Device: cuda
Loading model: knowledgator/SMILES2IUPAC-canonical-small on cuda
Processing rows 4640-4679 of 4681
  Checkpoint loaded: 4640 rows already done
  batch checkpoint saved (4680 rows total)

=== GPU batch rows 4680-4680 / 4680 ===


2026-06-15 17:46:02.171684: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-15 17:46:02.420089: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-06-15 17:46:02.420125: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-06-15 17:46:02.461798: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-15 17:46:03.282735: W tensorflow/stream_executor/platform/de

Device: cuda
Loading model: knowledgator/SMILES2IUPAC-canonical-small on cuda
Processing rows 4680-4680 of 4681
  Checkpoint loaded: 4680 rows already done
  batch checkpoint saved (4681 rows total)
Device: cuda


2026-06-15 17:46:12.952906: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-15 17:46:13.237674: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-06-15 17:46:13.237709: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-06-15 17:46:13.288956: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-15 17:46:14.107614: W tensorflow/stream_executor/platform/de

Loading model: knowledgator/SMILES2IUPAC-canonical-small on cuda
Processing 4681 rows with SMILES for reverse naming
  Checkpoint loaded: 4681 rows already done
Exported: dataset_from_ccast.xlsx
  IUPAC produced: 4554/4681
  Flagged: 426/4681


In [ ]:
# Sanity check — READ THIS before downloading
import pandas as pd

filled = df["STOUT_IUPAC"].notna() & df["STOUT_IUPAC"].astype(str).str.strip().replace("nan", "").ne("")
n_filled = int(filled.sum())
n_total = len(df)

print(f"Non-empty IUPAC: {n_filled}/{n_total}")
print()
print("Flag counts:")
print(df["Pass3_CCAST_Flag"].value_counts())
print()

if n_filled < n_total * 0.95:
    print("=" * 60)
    print("FAILED — most rows have no IUPAC (CUDA crash pattern).")
    print("Fix: Kernel -> Restart, set USE_CPU=True or GPU_BATCH=True, Run All.")
    print("Do NOT use old inline STOUT cells.")
    print("=" * 60)
else:
    print("SUCCESS — IUPAC generated for nearly all rows.")
    print("Many LOW_SIMILARITY flags are expected (1947 names vs modern IUPAC).")
    print(f"\nSaved: {OUTPUT}")
    print("Download dataset_from_ccast.xlsx to your PC for Streamlit curation.")

Non-empty IUPAC: 4554/4681

Flag counts:
OK                4255
SUSPICIOUS         293
CONVERT_FAILED     127
LOW_SIMILARITY       6
Name: Pass3_CCAST_Flag, dtype: int64

SUCCESS — IUPAC generated for nearly all rows.
Many LOW_SIMILARITY flags are expected (1947 names vs modern IUPAC).

Saved: dataset_from_ccast.xlsx
Download dataset_from_ccast.xlsx to your PC for Streamlit curation.


In [ ]:
# Preview
cols = ["Chemical", "SMILES", "STOUT_IUPAC", "JaroWinkler_Score", "Pass3_CCAST_Flag", "Pass3_CCAST_Error"]
cols = [c for c in cols if c in df.columns]
df[cols].head(10)

,Chemical,SMILES,STOUT_IUPAC,JaroWinkler_Score,Pass3_CCAST_Flag,Pass3_CCAST_Error
0,"Abietic acid, diethylene glycol ester (Flexalyn)",CC(C)C1=CC2=CC[C@@H]3[C@](C)(CCC[C@@]3(C)C(=O)...,"2-(2-hydroxyethoxy)ethyl 1,10-dimethyl-10-prop...",0.4821,OK,NaN
1,"Abietic acid, hydrogenated methyl ester (Herco...",COC(=O)[C@]1(C)CCC[C@]2(C)[C@H]3CCC(=CC3CC[C@@...,"methyl 1,1'-dimethyl-1'-propan-2-ylspiro[2,3,4...",0.5033,OK,NaN
2,Acetaldehyde dioctyl mercaptal,CCCCCCCCSC(C)SCCCCCCCC,1-(1-octylsulfanylethylsulfanyl)octane,0.5537,OK,NaN
3,"Abietic acid, ethyl ester",CCOC(=O)[C@]1(C)CCC[C@]2(C)[C@H]3CCC(=CC3=CC[C...,"ethyl 1,1'-dimethyl-1'-propan-2-ylspiro[2,3,4,...",0.4954,OK,NaN
4,"Abietic acid, methyl ester (Abalyn)",COC(=O)[C@]1(C)CCC[C@]2(C)[C@H]3CCC(=CC3=CC[C@...,"methyl 1,1'-dimethyl-1'-propan-2-ylspiro[2,3,4...",0.4838,OK,NaN
5,"Acetic acid, o-allyl-p-cresol ester",CC(O)=O.Cc1ccc(O)c(CC=C)c1,ethanol;2-methyl-3-prop-2-enylcyclopropan-1-ol,0.5744,OK,NaN
6,Acenaphthene,C1Cc2cccc3cccc1c23,"spiro[2-bicyclo[1.1.0]butane-2,1'-cyclopropane]",0.3759,SUSPICIOUS,NaN
7,Acetamide,CC(N)=O,2-aminoethanal,0.5873,OK,NaN
8,"Acetic acid, iso-amyl ester",CC(C)CCOC(C)=O,2-(3-methylbutoxy)propanal,0.4117,SUSPICIOUS,NaN
9,Acetanilide,CC(=O)Nc1ccccc1,"N-(1,2-dihydrooxireno[2,3-d]pyrimidin-4-yl)ace...",0.4502,OK,NaN
